# 8.2a All five CV criteria from configuration

This notebook runs **brightness, contrast, color palette, framing/subject position, and subject/background sharpness** through the actual pipeline. It displays measurements for all five, individual scoring availability, and the overall score.

The default image is the supplied synthetic fixture. Its central checkerboard has a known subject box `(0.25, 0.25, 0.75, 0.75)`. A clearly identified fixture provider passes this box through the pipeline's existing subject-selection interface. This is **manual test input, not automatic subject detection**. It permits an offline demonstration of every measurement without downloading detector weights.

For your own photograph, select the existing RT-DETR detector below. No VLM client, Ollama, BrandLens API server, or model endpoint is used.

In [1]:
import json
from functools import partial
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "brandlens").is_dir()
    and (path / "config" / "brand_standards.json").is_file()
)
    
    

In [2]:
from brandlens.config import load_cv_config
from brandlens.cv.subject.detector_base import (
    DetectionResult,
    SubjectCandidate,
    SubjectDetectorBase,
)
from brandlens.cv.subject.rtdetr_detector import RTDetrSubjectDetector
from brandlens.cv.subject.selector import SingleSubjectSelector
from brandlens.pipeline import score_photo
from brandlens.preprocessing.image_io import load_image

## Load the five-criterion CV JSON once

The JSON retains the original measurement settings and provisional/unset scoring bounds. All five measurements are supported. Palette/framing/sharpness scores remain unavailable where thresholds are unset; framing/sharpness scoring also needs alignment with the CV measurement names. The notebook keeps these reasons visible instead of inventing scores.

For real object detection, install the existing optional dependencies with `pip install -e ".[subject]"`, set `IMAGE_PATH` to your photo, and select `SUBJECT_MODE = "rtdetr"`. RT-DETR runs locally and may download weights on first use. No/ambiguous primary subjects remain explicit review cases.

In [3]:
SAMPLE_PATH = ROOT / "examples" / "data" / "backend_scoring_sample.png"
IMAGE_PATH = SAMPLE_PATH
CONFIG_PATH = ROOT / "config" / "brand_standards_cv_only_test.json"
SCHEMA_PATH = ROOT / "config" / "brand_standards.schema.json"
SHOW_MEASUREMENTS = True

# "fixture": known subject box for the supplied synthetic image; no model download.
# "rtdetr": actual object detection for your photograph; needs the subject dependencies.
SUBJECT_MODE = "fixture"

configuration = load_cv_config(CONFIG_PATH, schema_path=SCHEMA_PATH)

In [4]:
class DemoFixtureDetector(SubjectDetectorBase):
    """Supply the known central ROI of the demo image; performs no detection."""

    def detect(self, image):
        return DetectionResult(
            candidates=(
                SubjectCandidate(0.25, 0.25, 0.75, 0.75, "synthetic_subject", 1.0),
            ),
            source="manual_demo_fixture",
            source_version="1",
            image_size=image.size,
        )


if SUBJECT_MODE == "fixture":
    if IMAGE_PATH.resolve() != SAMPLE_PATH.resolve():
        raise ValueError(
            "The fixture box belongs to the supplied synthetic image. "
            "Select SUBJECT_MODE='rtdetr' for another photograph."
        )
    subject_detector = DemoFixtureDetector()
elif SUBJECT_MODE == "rtdetr":
    subject_detector = RTDetrSubjectDetector()
else:
    raise ValueError("SUBJECT_MODE must be 'fixture' or 'rtdetr'.")

subject_selector = SingleSubjectSelector()
score = partial(
    score_photo,
    subject_detector=subject_detector,
    subject_selector=subject_selector,
)

In [5]:
# The detector/fixture provider runs once; framing and sharpness share its ROI.
with load_image(IMAGE_PATH) as image:
    scored, cv_results, _ = score(image, configuration)

In [10]:
print(f"Scoring results for {IMAGE_PATH.name}:")
overall = pd.DataFrame([{
    "rubric_id": scored.rubric_id,
    "overall_score": scored.overall_score,
    "status": scored.status,
    "measured_criteria": sum(row["status"] == "available" for row in cv_results.values()),
    "scored_criteria": sum(row.availability == "ok" for row in scored.criteria),
    "configured_criteria": len(scored.criteria),
}])
display(overall)

individual = pd.DataFrame([{
    "criterion_id": row.criterion_id,
    "measurement_status": cv_results[row.criterion_id]["status"],
    "score": row.score,
    "applied_weight": row.weight,
    "scoring_availability": row.availability,
    "reason": "; ".join(row.reasons),
} for row in scored.criteria])

print("All five CV criteria:")
display(individual)

Scoring results for backend_scoring_sample.png:


,rubric_id,overall_score,status,measured_criteria,scored_criteria,configured_criteria
0,all-five-cv-demo,100.0,partial,5,2,5


All five CV criteria:


,criterion_id,measurement_status,score,applied_weight,scoring_availability,reason
0,exposure_brightness,available,100.0,0.5,ok,
1,contrast,available,100.0,0.5,ok,
2,color_palette_fit,available,NaN,0.0,unavailable_needs_review,Rubric threshold unset: palette_distance_max
3,framing_subject_position,available,NaN,0.0,unavailable_needs_review,"Rubric thresholds unset: center_offset_max, th..."
4,sharpness_subject_emphasis,available,NaN,0.0,unavailable_needs_review,Rubric thresholds unset: subject_sharpness_min...


In [11]:
if SHOW_MEASUREMENTS:
    rows = [{
        "criterion_id": cid,
        "measurement": name,
        "value": json.dumps(value) if isinstance(value, (list, dict)) else value,
        "unit": result["units"].get(name, ""),
    } for cid, result in cv_results.items()
      for name, value in result["measurements"].items()]
    measurements = pd.DataFrame(
        rows, columns=["criterion_id", "measurement", "value", "unit"]
    )
    
    print("CV measurements:")
    display(measurements)

CV measurements:


,criterion_id,measurement,value,unit
0,exposure_brightness,mean_luminance,99.0,grayscale_0_255
1,exposure_brightness,near_black_fraction,0.0,fraction_0_1
2,exposure_brightness,near_white_fraction,0.0,fraction_0_1
3,contrast,luminance_std,58.122715,grayscale_0_255
4,contrast,p5_p95_range,163.0,grayscale_0_255
5,contrast,near_black_fraction,0.0,fraction_0_1
6,contrast,near_white_fraction,0.0,fraction_0_1
7,color_palette_fit,dominant_colors,"[{""rgb"": [50, 70, 100], ""hex"": ""#324664"", ""lab...",RGB_0_255_hex_CIELAB_D65_proportion_0_1
8,color_palette_fit,palette_distance,14.318438,delta_e_76
9,framing_subject_position,subject_center_x,0.5,fraction


The fixture's five measurement statuses should be `available`. Its overall scoring status is expected to be `partial`: only criteria with currently usable scoring rules and configured bounds contribute to that score. The coverage columns distinguish **measured** criteria from **scored** criteria.

To inspect all native outputs, expand `cv_results`. Framing and sharpness results include their shared subject-region provenance. The original brand rubric and source implementations are unchanged. This notebook has no unit-test loops.